# Machine Failure Prediction on the AI4I 2020 Dataset

**Objective:** predict whether a machine will fail from its operating conditions, so that maintenance can be planned before a breakdown.

**Data:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository, S. Matzka, 2020). 10,000 synthetic records with air temperature, process temperature, rotational speed, torque, tool wear and product quality (L, M, H). The target is `Machine failure` (0 or 1).

**Approach:** data quality checks and exploration, leakage-free feature preparation, a stratified train/test split, a class-weighted Random Forest evaluated with precision, recall and F1, and a SQL analysis in SQLite. The saved model powers the Flask app in this repository.

## 1. Data loading and quality checks

In [ ]:
import pandas as pd

df = pd.read_csv('ai4i2020.csv')
print('Shape:', df.shape)
print('Missing values:', int(df.isnull().sum().sum()))
print(df['Machine failure'].value_counts())
print('Failure rate (%):', round(df['Machine failure'].mean() * 100, 2))

The data has no missing values. Only 3.39% of records are failures, so the classes are heavily imbalanced. Plain accuracy would be misleading here: a model that always predicts "no failure" is already about 96.6% accurate. Precision, recall and F1 on the failure class are used instead.

## 2. Exploratory analysis

In [ ]:
print(df.groupby('Type')['Machine failure'].agg(['count', 'sum', 'mean']))
print(df.describe().round(2))

Failure rate by product quality: L (low) 3.92%, M (medium) 2.77%, H (high) 2.09%. Lower quality variants fail more often.

## 3. Feature preparation

`Machine failure` is defined as 1 whenever any of five failure modes occurs: TWF (tool wear failure), HDF (heat dissipation failure), PWF (power failure), OSF (overstrain failure) or RNF (random failure). Using those columns as inputs would leak the answer into the model, so they are excluded along with the identifier columns `UDI` and `Product ID`. The categorical `Type` column is one-hot encoded.

In [ ]:
X = df.drop(columns=['UDI', 'Product ID', 'Machine failure',
                     'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df['Machine failure']
X = pd.get_dummies(X, columns=['Type'], drop_first=True, dtype=int)

assert 'TWF' not in X.columns   # no leakage columns among the inputs
print(X.columns.tolist())
print(X.shape)

## 4. Train/test split

80% of the data is used for training and 20% is held out for testing. The split is stratified so both parts keep the same failure rate, and `random_state=42` makes the result reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print('Failure % in train:', round(y_train.mean() * 100, 2),
      '| in test:', round(y_test.mean() * 100, 2))

## 5. Model: Random Forest with class weighting

A Random Forest averages many decision trees trained on random samples of rows and features. It handles non-linear relationships and needs no feature scaling. `class_weight='balanced'` penalises mistakes on the rare failure class more heavily, which counters the imbalance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

## 6. Evaluation

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score)

pred = rf.predict(X_test)
print('Baseline accuracy (always predict no failure):', round(1 - y_test.mean(), 4))
print('Confusion matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))
print('Failure class -> precision:', round(precision_score(y_test, pred), 3),
      '| recall:', round(recall_score(y_test, pred), 3),
      '| F1:', round(f1_score(y_test, pred), 3))

Reading the matrix: the top-right cell is false alarms, the bottom-left is missed failures, the bottom-right is failures caught. In maintenance, a missed failure usually costs more than an unnecessary inspection, so recall matters most; precision limits the number of false alarms.

## 7. Feature importance

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.round(3))

Torque, rotational speed and tool wear are the most influential inputs, which agrees with how the failure modes are defined (overstrain, power and tool wear).

## 8. SQL analysis (SQLite)

The records are loaded into a SQLite table with clean column names, and queried with GROUP BY, HAVING, CASE and subqueries.

In [ ]:
import sqlite3

db = df.rename(columns={
    'UDI': 'udi', 'Product ID': 'product_id', 'Type': 'type',
    'Air temperature [K]': 'air_temp', 'Process temperature [K]': 'process_temp',
    'Rotational speed [rpm]': 'rot_speed', 'Torque [Nm]': 'torque',
    'Tool wear [min]': 'tool_wear', 'Machine failure': 'failure',
    'TWF': 'twf', 'HDF': 'hdf', 'PWF': 'pwf', 'OSF': 'osf', 'RNF': 'rnf'})

conn = sqlite3.connect('machines.db')
db.to_sql('machines', conn, if_exists='replace', index=False)

def q(sql):
    return pd.read_sql_query(sql, conn)

# Failure rate by product type (GROUP BY + HAVING)
q('''
SELECT type, COUNT(*) AS machines, SUM(failure) AS failures,
       ROUND(100.0 * SUM(failure) / COUNT(*), 2) AS failure_rate_pct
FROM machines
GROUP BY type
HAVING COUNT(*) > 500
ORDER BY failure_rate_pct DESC
''')

In [ ]:
# Failures with above-average tool wear (subquery)
q('''
SELECT COUNT(*) AS failures_with_high_wear
FROM machines
WHERE failure = 1
  AND tool_wear > (SELECT AVG(tool_wear) FROM machines)
''')

In [ ]:
# Failure rate by tool-wear band (CASE + GROUP BY)
q('''
SELECT CASE WHEN tool_wear < 100 THEN '1 low (<100)'
            WHEN tool_wear < 200 THEN '2 medium (100-199)'
            ELSE '3 high (200+)' END AS wear_band,
       COUNT(*) AS machines, SUM(failure) AS failures,
       ROUND(100.0 * SUM(failure) / COUNT(*), 2) AS failure_rate_pct
FROM machines
GROUP BY wear_band
ORDER BY wear_band
''')

**Key finding:** the failure rate is about 2.2% below 100 minutes of tool wear and 2.5% between 100 and 199 minutes, but jumps to 15.4% at 200 minutes or more. Replacing tools before they reach about 200 minutes would remove a large share of failures.

In [ ]:
# Failures by failure mode
q('''
SELECT SUM(twf) AS tool_wear, SUM(hdf) AS heat_dissipation,
       SUM(pwf) AS power, SUM(osf) AS overstrain, SUM(rnf) AS random_fail
FROM machines
''')

In [ ]:
# Product types whose failure rate is above the overall rate (HAVING + subquery)
q('''
SELECT type, ROUND(100.0 * AVG(failure), 2) AS failure_rate_pct
FROM machines
GROUP BY type
HAVING AVG(failure) > (SELECT AVG(failure) FROM machines)
''')

## 9. Save the model

In [ ]:
import joblib, sklearn

joblib.dump({'model': rf, 'columns': list(X.columns),
             'sklearn_version': sklearn.__version__}, 'model.joblib')
print('Saved model.joblib')

## 10. Limitations and next steps

- The dataset is synthetic, so the results show the method rather than real plant performance.
- The model uses default Random Forest settings with no hyperparameter tuning.
- The decision threshold is the default 0.5. Lowering it would raise recall at the cost of more false alarms.
- Next steps: tune the threshold using a precision-recall curve, compare gradient boosting, validate with cross-validation, and log predictions from the web app.